# VCF Training Development

Random Forest training and predictions.

## 1. Import Libraries

In [ ]:
# !pip install shap

In [ ]:
# import sys
# import getpass

# this assumes you are using the dev kernel
# sys.path.append(f'/home/{getpass.getuser()}/.local/lib/python3.12/site-packages')

In [ ]:
import os
import cudf
import cuml
import dask
import shap
import getpass
import logging
import dask_cudf
import xarray as xr
import pandas as pd
import rioxarray as rxr
import dask.dataframe as dd
import matplotlib.pyplot as plt
from osgeo import gdal
from dask.distributed import Client
from pathlib import Path
from dask_cuda import LocalCUDACluster
from cuml.dask.ensemble import RandomForestRegressor as cumlDaskRF
from cuml.metrics import accuracy_score, confusion_matrix
from matplotlib.colors import LinearSegmentedColormap

import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    mean_absolute_percentage_error,
    median_absolute_error,
    max_error,
    explained_variance_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from glob import glob
from joblib import dump, load

import warnings
warnings.simplefilter("ignore", FutureWarning)

In [ ]:
print(f"cuML version: {cuml.__version__}")
print(f"Dask version: {dask.__version__}")
print(f"Dask-cuDF version: {dask_cudf.__version__}")
print(f"cuDF version: {cudf.__version__}")

## 2. Initialize Dask Cluster

In [ ]:
# setup cluster
cluster = LocalCUDACluster()
client = Client(cluster)
workers = client.has_what().keys()
n_workers = len(workers)
cluster

In [ ]:
print(f"Dask cluster has {n_workers} workers")

## 3. Set Features to Work With (The only place where we need to modify things for testing)

In [ ]:
# regex to find all training parquet filenames
# parquet_files_regex = '/explore/nobackup/projects/ilab/projects/MODIS-VCF/processedTiles/MOD44C/training-V5.0.3/h31v11-2019.parq'
# parquet_files_regex = '/explore/nobackup/projects/ilab/projects/MODIS-VCF/processedTiles/MOD44C/training-V5.0.3/*.parq'
# parquet_files_regex = '/explore/nobackup/projects/ilab/scratch/mcarrol2/notebooks/vcf_clustering/new_parq/*.parq'
parquet_files_regex = '/explore/nobackup/projects/ilab/scratch/mcarrol2/MOD44B_testing/VCF_code/parq_test/*.parq'

In [ ]:
# where the model and intermediate data will be stored
# output_dir = '/explore/nobackup/projects/ilab/projects/MODIS-VCF/random-forest-development'
output_dir = f'/explore/nobackup/projects/ilab/scratch/{getpass.getuser()}/notebooks/vcf_rf_test'

# filenames to store intermediate training and test datasets
train_dataset_filename = os.path.join(output_dir, 'VCF-Train-Dataset-bare.parquet')
test_dataset_filename = os.path.join(output_dir, 'VCF-Test-Dataset-bare.parquet')

# parameters for the training and test datasets
train_dataset_ratio = 0.80
target_col = 'PercentTree'
output_model_filename = os.path.join(output_dir, 'random_forest_model.joblib')

# path for raster predictions
raster_data_path = '/explore/nobackup/projects/ilab/projects/MODIS-VCF/processedTiles/MOD44C/h20v06/2019/3-Metrics/*.tif'

In [ ]:
# random forest parameters (defaults, feel free to modify)
"""
gpu_rf_params = {
    "n_estimators": 100,  # Number of trees in the forest
    "max_depth": None,  # Maximum depth of each tree
    "min_samples_split": 2,  # Minimum samples required to split an internal node
    "min_samples_leaf": 1,  # Minimum samples required to be a leaf node
    "min_weight_fraction_leaf": 0.0,  # Minimum weighted fraction of sum total of weights to be a leaf
    "max_features": "sqrt",  # Number of features to consider for best split ('auto', 'sqrt', 'log2', or int/float)
    "max_leaf_nodes": None,  # Maximum number of leaf nodes
    "min_impurity_decrease": 0.0,  # Threshold to grow a node only if impurity decrease is greater
    "bootstrap": True,  # Whether bootstrap samples are used
    "oob_score": False,  # Whether to use out-of-bag samples to estimate generalization
    "random_state": None,  # Seed for reproducibility
    "verbose": 0,  # Controls verbosity
    "warm_start": False,  # Whether to reuse previous solution
    "max_samples": None,  # Number of samples to draw when bootstrap=True
}
"""
rf_params = {
    "n_estimators": 100,  # Number of trees in the forest
    "criterion": "squared_error",  # Function to measure split quality ('squared_error', 'absolute_error', 'friedman_mse', 'poisson')
    "max_depth": None,  # Maximum depth of each tree
    "min_samples_split": 2,  # Minimum samples required to split an internal node
    "min_samples_leaf": 1,  # Minimum samples required to be a leaf node
    "min_weight_fraction_leaf": 0.0,  # Minimum weighted fraction of sum total of weights to be a leaf
    "max_features": "sqrt",  # Number of features to consider for best split ('auto', 'sqrt', 'log2', or int/float)
    "max_leaf_nodes": None,  # Maximum number of leaf nodes
    "min_impurity_decrease": 0.0,  # Threshold to grow a node only if impurity decrease is greater
    "bootstrap": True,  # Whether bootstrap samples are used
    "oob_score": False,  # Whether to use out-of-bag samples to estimate generalization
    "n_jobs": -1,  # Number of parallel jobs (-1 uses all processors)
    "random_state": None,  # Seed for reproducibility
    "verbose": 0,  # Controls verbosity
    "warm_start": False,  # Whether to reuse previous solution
    "ccp_alpha": 0.0,  # Complexity parameter for pruning
    "max_samples": None,  # Number of samples to draw when bootstrap=True
}

In [ ]:
# columns names to choose from
# NOTE: comment out the ones you do not want load from the dataset
# Leave the target column (PercentTree) in the list (this will be excluded when training)
col_names = [
    #'tid-year',
    #'x',
    #'y',
    'PercentTree',
    #'AmpBandRefl-Band_1',
    #'AmpBandRefl-Band_2',
    #'AmpBandRefl-Band_3',
    #'AmpBandRefl-Band_4',
    #'AmpBandRefl-Band_5',
    #'AmpBandRefl-Band_6',
    #'AmpBandRefl-Band_7',
    #'AmpBandRefl-NDVI',
    #'AmpGreenestBandRefl-Band_1',
    #'AmpGreenestBandRefl-Band_2',
    #'AmpGreenestBandRefl-Band_3',
    #'AmpGreenestBandRefl-Band_4',
    #'AmpGreenestBandRefl-Band_5',
    #'AmpGreenestBandRefl-Band_6',
    #'AmpGreenestBandRefl-Band_7',
    #'AmpGreenestBandRefl-NDVI',
    #'AmpWarmestBandRefl-Band_1',
    #'AmpWarmestBandRefl-Band_2',
    #'AmpWarmestBandRefl-Band_3',
    #'AmpWarmestBandRefl-Band_4',
    #'AmpWarmestBandRefl-Band_5',
    #'AmpWarmestBandRefl-Band_6',
    #'AmpWarmestBandRefl-Band_7',
    #'AmpWarmestBandRefl-NDVI',
    #'BandReflMax-Band_1',
    #'BandReflMax-Band_2',
    #'BandReflMax-Band_3',
    #'BandReflMax-Band_4',
    #'BandReflMax-Band_5',
    #'BandReflMax-Band_6',
    'BandReflMax-Band_7',
    #'BandReflMax-NDVI',
    #'BandReflMaxGreenness-Band_1',
    #'BandReflMaxGreenness-Band_2',
    #'BandReflMaxGreenness-Band_3',
    #'BandReflMaxGreenness-Band_4',
    #'BandReflMaxGreenness-Band_5',
    #'BandReflMaxGreenness-Band_6',
    #'BandReflMaxGreenness-Band_7',
    'BandReflMaxGreenness-NDVI',
    #'BandReflMaxTemp-Band_1',
    #'BandReflMaxTemp-Band_2',
    #'BandReflMaxTemp-Band_3',
    #'BandReflMaxTemp-Band_4',
    #'BandReflMaxTemp-Band_5',
    #'BandReflMaxTemp-Band_6',
    #'BandReflMaxTemp-Band_7',
    'BandReflMaxTemp-NDVI',
    #'BandReflMedian-Band_1',
    #'BandReflMedian-Band_2',
    #'BandReflMedian-Band_3',
    #'BandReflMedian-Band_4',
    #'BandReflMedian-Band_5',
    #'BandReflMedian-Band_6',
    'BandReflMedian-Band_7',
    'BandReflMedian-NDVI',
    #'BandReflMedianGreenness-Band_1',
    #'BandReflMedianGreenness-Band_2',
    #'BandReflMedianGreenness-Band_3',
    #'BandReflMedianGreenness-Band_4',
    #'BandReflMedianGreenness-Band_5',
    #'BandReflMedianGreenness-Band_6',
    'BandReflMedianGreenness-Band_7',
    'BandReflMedianGreenness-NDVI',
    #'BandReflMedianTemp-Band_1',
    #'BandReflMedianTemp-Band_2',
    #'BandReflMedianTemp-Band_3',
    #'BandReflMedianTemp-Band_4',
    #'BandReflMedianTemp-Band_5',
    #'BandReflMedianTemp-Band_6',
    #'BandReflMedianTemp-Band_7',
    'BandReflMedianTemp-NDVI',
    #'BandReflMin-Band_1',
    #'BandReflMin-Band_2',
    #'BandReflMin-Band_3',
    #'BandReflMin-Band_4',
    #'BandReflMin-Band_5',
    #'BandReflMin-Band_6',
    'BandReflMin-Band_7',
    'BandReflMin-NDVI',
    #'BandReflMinGreenness-Band_1',
    #'BandReflMinGreenness-Band_2',
    #'BandReflMinGreenness-Band_3',
    #'BandReflMinGreenness-Band_4',
    #'BandReflMinGreenness-Band_5',
    #'BandReflMinGreenness-Band_6',
    #'BandReflMinGreenness-Band_7',
    'BandReflMinGreenness-NDVI',
    #'BandReflMinTemp-Band_1',
    #'BandReflMinTemp-Band_2',
    #'BandReflMinTemp-Band_3',
    #'BandReflMinTemp-Band_4',
    #'BandReflMinTemp-Band_5',
    #'BandReflMinTemp-Band_6',
    #'BandReflMinTemp-Band_7',
    'BandReflMinTemp-NDVI',
    #'Greenest3MeanBandRefl-Band_1',
    #'Greenest3MeanBandRefl-Band_2',
    'Greenest3MeanBandRefl-Band_3',
    #'Greenest3MeanBandRefl-Band_4',
    #'Greenest3MeanBandRefl-Band_5',
    #'Greenest3MeanBandRefl-Band_6',
    'Greenest3MeanBandRefl-Band_7',
    'Greenest3MeanBandRefl-NDVI',
    #'Greenest6MeanBandRefl-Band_1',
    #'Greenest6MeanBandRefl-Band_2',
    'Greenest6MeanBandRefl-Band_3',
    #'Greenest6MeanBandRefl-Band_4',
    #'Greenest6MeanBandRefl-Band_5',
    #'Greenest6MeanBandRefl-Band_6',
    'Greenest6MeanBandRefl-Band_7',
    'Greenest6MeanBandRefl-NDVI',
    #'Greenest8MeanBandRefl-Band_1',
    #'Greenest8MeanBandRefl-Band_2',
    'Greenest8MeanBandRefl-Band_3',
    #'Greenest8MeanBandRefl-Band_4',
    #'Greenest8MeanBandRefl-Band_5',
    #'Greenest8MeanBandRefl-Band_6',
    #'Greenest8MeanBandRefl-Band_7',
    'Greenest8MeanBandRefl-NDVI',
    #'Lowest3MeanBandRefl-Band_1',
    #'Lowest3MeanBandRefl-Band_2',
    'Lowest3MeanBandRefl-Band_3',
    #'Lowest3MeanBandRefl-Band_4',
    #'Lowest3MeanBandRefl-Band_5',
    #'Lowest3MeanBandRefl-Band_6',
    #'Lowest3MeanBandRefl-Band_7',
    #'Lowest6MeanBandRefl-Band_1',
    #'Lowest6MeanBandRefl-Band_2',
    #'Lowest6MeanBandRefl-Band_3',
    #'Lowest6MeanBandRefl-Band_4',
    #'Lowest6MeanBandRefl-Band_5',
    #'Lowest6MeanBandRefl-Band_6',
    #'Lowest6MeanBandRefl-Band_7',
    #'Lowest8MeanBandRefl-Band_1',
    #'Lowest8MeanBandRefl-Band_2',
    'Lowest8MeanBandRefl-Band_3',
    #'Lowest8MeanBandRefl-Band_4',
    #'Lowest8MeanBandRefl-Band_5',
    #'Lowest8MeanBandRefl-Band_6',
    #'Lowest8MeanBandRefl-Band_7',
    #'TempMeanGreenest3',
    'TempMeanWarmest3',
    #'UnsortedMonthlyBands-Band_1-Day-2019065',
    #'UnsortedMonthlyBands-Band_1-Day-2019097',
    #'UnsortedMonthlyBands-Band_1-Day-2019129',
    #'UnsortedMonthlyBands-Band_1-Day-2019161',
    #'UnsortedMonthlyBands-Band_1-Day-2019193',
    #'UnsortedMonthlyBands-Band_1-Day-2019225',
    #'UnsortedMonthlyBands-Band_1-Day-2019257',
    #'UnsortedMonthlyBands-Band_1-Day-2019289',
    #'UnsortedMonthlyBands-Band_1-Day-2019321',
    #'UnsortedMonthlyBands-Band_1-Day-2019353',
    #'UnsortedMonthlyBands-Band_1-Day-2020017',
    #'UnsortedMonthlyBands-Band_1-Day-2020049',
    #'UnsortedMonthlyBands-Band_2-Day-2019065',
    #'UnsortedMonthlyBands-Band_2-Day-2019097',
    #'UnsortedMonthlyBands-Band_2-Day-2019129',
    #'UnsortedMonthlyBands-Band_2-Day-2019161',
    #'UnsortedMonthlyBands-Band_2-Day-2019193',
    #'UnsortedMonthlyBands-Band_2-Day-2019225',
    #'UnsortedMonthlyBands-Band_2-Day-2019257',
    #'UnsortedMonthlyBands-Band_2-Day-2019289',
    #'UnsortedMonthlyBands-Band_2-Day-2019321',
    #'UnsortedMonthlyBands-Band_2-Day-2019353',
    #'UnsortedMonthlyBands-Band_2-Day-2020017',
    #'UnsortedMonthlyBands-Band_2-Day-2020049',
    #'UnsortedMonthlyBands-Band_3-Day-2019065',
    #'UnsortedMonthlyBands-Band_3-Day-2019097',
    #'UnsortedMonthlyBands-Band_3-Day-2019129',
    #'UnsortedMonthlyBands-Band_3-Day-2019161',
    #'UnsortedMonthlyBands-Band_3-Day-2019193',
    #'UnsortedMonthlyBands-Band_3-Day-2019225',
    #'UnsortedMonthlyBands-Band_3-Day-2019257',
    #'UnsortedMonthlyBands-Band_3-Day-2019289',
    #'UnsortedMonthlyBands-Band_3-Day-2019321',
    #'UnsortedMonthlyBands-Band_3-Day-2019353',
    #'UnsortedMonthlyBands-Band_3-Day-2020017',
    #'UnsortedMonthlyBands-Band_3-Day-2020049',
    #'UnsortedMonthlyBands-Band_4-Day-2019065',
    #'UnsortedMonthlyBands-Band_4-Day-2019097',
    #'UnsortedMonthlyBands-Band_4-Day-2019129',
    #'UnsortedMonthlyBands-Band_4-Day-2019161',
    #'UnsortedMonthlyBands-Band_4-Day-2019193',
    #'UnsortedMonthlyBands-Band_4-Day-2019225',
    #'UnsortedMonthlyBands-Band_4-Day-2019257',
    #'UnsortedMonthlyBands-Band_4-Day-2019289',
    #'UnsortedMonthlyBands-Band_4-Day-2019321',
    #'UnsortedMonthlyBands-Band_4-Day-2019353',
    #'UnsortedMonthlyBands-Band_4-Day-2020017',
    #'UnsortedMonthlyBands-Band_4-Day-2020049',
    #'UnsortedMonthlyBands-Band_5-Day-2019065',
    #'UnsortedMonthlyBands-Band_5-Day-2019097',
    #'UnsortedMonthlyBands-Band_5-Day-2019129',
    #'UnsortedMonthlyBands-Band_5-Day-2019161',
    #'UnsortedMonthlyBands-Band_5-Day-2019193',
    #'UnsortedMonthlyBands-Band_5-Day-2019225',
    #'UnsortedMonthlyBands-Band_5-Day-2019257',
    #'UnsortedMonthlyBands-Band_5-Day-2019289',
    #'UnsortedMonthlyBands-Band_5-Day-2019321',
    #'UnsortedMonthlyBands-Band_5-Day-2019353',
    #'UnsortedMonthlyBands-Band_5-Day-2020017',
    #'UnsortedMonthlyBands-Band_5-Day-2020049',
    #'UnsortedMonthlyBands-Band_6-Day-2019065',
    #'UnsortedMonthlyBands-Band_6-Day-2019097',
    #'UnsortedMonthlyBands-Band_6-Day-2019129',
    #'UnsortedMonthlyBands-Band_6-Day-2019161',
    #'UnsortedMonthlyBands-Band_6-Day-2019193',
    #'UnsortedMonthlyBands-Band_6-Day-2019225',
    #'UnsortedMonthlyBands-Band_6-Day-2019257',
    #'UnsortedMonthlyBands-Band_6-Day-2019289',
    #'UnsortedMonthlyBands-Band_6-Day-2019321',
    #'UnsortedMonthlyBands-Band_6-Day-2019353',
    #'UnsortedMonthlyBands-Band_6-Day-2020017',
    #'UnsortedMonthlyBands-Band_6-Day-2020049',
    #'UnsortedMonthlyBands-Band_7-Day-2019065',
    #'UnsortedMonthlyBands-Band_7-Day-2019097',
    #'UnsortedMonthlyBands-Band_7-Day-2019129',
    #'UnsortedMonthlyBands-Band_7-Day-2019161',
    #'UnsortedMonthlyBands-Band_7-Day-2019193',
    #'UnsortedMonthlyBands-Band_7-Day-2019225',
    #'UnsortedMonthlyBands-Band_7-Day-2019257',
    #'UnsortedMonthlyBands-Band_7-Day-2019289',
    #'UnsortedMonthlyBands-Band_7-Day-2019321',
    #'UnsortedMonthlyBands-Band_7-Day-2019353',
    #'UnsortedMonthlyBands-Band_7-Day-2020017',
    #'UnsortedMonthlyBands-Band_7-Day-2020049',
    #'UnsortedMonthlyBands-NDVI-Day-2019065',
    #'UnsortedMonthlyBands-NDVI-Day-2019097',
    #'UnsortedMonthlyBands-NDVI-Day-2019129',
    #'UnsortedMonthlyBands-NDVI-Day-2019161',
    #'UnsortedMonthlyBands-NDVI-Day-2019193',
    #'UnsortedMonthlyBands-NDVI-Day-2019225',
    #'UnsortedMonthlyBands-NDVI-Day-2019257',
    #'UnsortedMonthlyBands-NDVI-Day-2019289',
    #'UnsortedMonthlyBands-NDVI-Day-2019321',
    #'UnsortedMonthlyBands-NDVI-Day-2019353',
    #'UnsortedMonthlyBands-NDVI-Day-2020017',
    #'UnsortedMonthlyBands-NDVI-Day-2020049',
    #'Warmest3MeanBandRefl-Band_1',
    #'Warmest3MeanBandRefl-Band_2',
    #'Warmest3MeanBandRefl-Band_3',
    #'Warmest3MeanBandRefl-Band_4',
    #'Warmest3MeanBandRefl-Band_5',
    #'Warmest3MeanBandRefl-Band_6',
    #'Warmest3MeanBandRefl-Band_7',
    'Warmest3MeanBandRefl-NDVI',
    #'Warmest6MeanBandRefl-Band_1',
    #'Warmest6MeanBandRefl-Band_2',
    #'Warmest6MeanBandRefl-Band_3',
    #'Warmest6MeanBandRefl-Band_4',
    #'Warmest6MeanBandRefl-Band_5',
    #'Warmest6MeanBandRefl-Band_6',
    #'Warmest6MeanBandRefl-Band_7',
    'Warmest6MeanBandRefl-NDVI',
    #'Warmest8MeanBandRefl-Band_1',
    #'Warmest8MeanBandRefl-Band_2',
    #'Warmest8MeanBandRefl-Band_3',
    #'Warmest8MeanBandRefl-Band_4',
    #'Warmest8MeanBandRefl-Band_5',
    #'Warmest8MeanBandRefl-Band_6',
    #'Warmest8MeanBandRefl-Band_7',
    'Warmest8MeanBandRefl-NDVI'
]

In [ ]:
os.makedirs(output_dir, exist_ok=True)

## 4. Generate Dataset if Not Already on the Filesystem (No need to modify anything after here)

In [ ]:
# Function for stratified sampling
def stratified_split(sub_df, train_ratio=0.80):
    sub_df = sub_df.sample(frac=1, random_state=42)  # Shuffle within group
    split_idx = int(len(sub_df) * train_ratio)
    return sub_df[:split_idx], sub_df[split_idx:]

# Generate training and test datasets
def generate_train_test_datasets(
            dask_gpu_df, 
            train_dataset_filename: str,
            test_dataset_filename: str,
            target_col: str = 'PercentTree',
            train_ratio: float = 0.80
        ):
    
    # Apply stratified split per class
    grouped = dask_gpu_df.groupby(target_col).apply(stratified_split)#, meta=dask_gpu_df)
    train, test = zip(*grouped)

    # Convert lists of Dask-cuDF partitions into DataFrames
    train = pd.concat(train)
    test = pd.concat(test)
    
    return train, test

In [ ]:
# only run if the dataset does not exist
if not os.path.exists(train_dataset_filename) or not os.path.exists(test_dataset_filename):
    
    # Get a list of all Parquet files in the directory
    parquet_files = glob(parquet_files_regex)

    # Initialize an empty list to hold the DataFrames
    dfs = []

    # Iterate over each file, read it and append it to the list
    for filename in parquet_files:
        print(filename)
        df = pd.read_parquet(filename)
        dfs.append(df)

    # Concatenate all DataFrames into one
    dask_df = pd.concat(dfs, ignore_index=True)
    print("Done concatenating")
    
    # generate and save dataset on disk
    train_df, test_df = generate_train_test_datasets(
        dask_df, 
        train_dataset_filename,
        test_dataset_filename,
        target_col,
        train_dataset_ratio
    )
    print(train_df.shape, test_df.shape)

    # save dataset to disk so this step does not need to be repeated
    train_df.to_parquet(train_dataset_filename)
    test_df.to_parquet(test_dataset_filename)
    print("Done saving intermediate dataset")

## 5. Load dataset if it's already available

In [ ]:
# GPU Version
# train_cudf = dask_cudf.read_parquet(train_dataset_filename, columns=col_names, blocksize="512MB")
# train_cudf = train_cudf.repartition(npartitions=n_workers)
# test_cudf = dask_cudf.read_parquet(test_dataset_filename, columns=col_names, blocksize="512MB")
# test_cudf = test_cudf.repartition(npartitions=n_workers)
# print(f"Train Dataset Size: {train_cudf.shape}, Test Dataset Size: {test_cudf.shape}")

# CPU Version
train_df = pd.read_parquet(train_dataset_filename, columns=col_names)
test_df = pd.read_parquet(test_dataset_filename, columns=col_names)
print(f"Train Dataset Size: {train_df.shape}, Test Dataset Size: {test_df.shape}")

In [ ]:
# plot the distribution of training and test
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)

# Plot histograms
axes[0].hist(train_df[target_col], bins=range(min(train_df[target_col]), max(train_df[target_col]) + 2), edgecolor='black', alpha=0.7)
axes[1].hist(test_df[target_col], bins=range(min(test_df[target_col]), max(test_df[target_col]) + 2), edgecolor='black', alpha=0.7)

# Titles and labels
axes[0].set_title(f'Histogram of Train Dataset: {target_col}')
axes[1].set_title(f'Histogram of Test Dataset: {target_col}')

for ax in axes:
    ax.set_xlabel('Value')
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# split the datasets into X and y
X_train, y_train = train_df.drop([target_col], axis=1).astype('float32'), train_df[target_col].astype('int32')
X_test, y_test = test_df.drop([target_col], axis=1).astype('float32'), test_df[target_col].astype('int32')
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

## 6. Setup and Train Random Forest

In [ ]:
"""
print(dask_df.shape, dask_df.columns)

    print(dask_df.npartitions)

    X = dask_df.drop(columns=['PercentTree']).astype('float32')  # Adjust column names
    y = dask_df['PercentTree'].astype('int32')

    # Initialize cuML's Dask-based Random Forest
    rf = cumlDaskRF(n_estimators=100, max_depth=10, ignore_empty_partitions=True)  # Tune hyperparameters as needed

    # Train the model (lazy execution)
    rf.fit(X, y)

    # Run multi-GPU inference
    y_pred = rf.predict(X)

    # Convert predictions to a Dask-cuDF DataFrame
    y_pred = y_pred.compute()  # Gather results from all GPUs to a single GPU
    print(y_pred)
"""
# Initialize and train the model
# rf = cumlDaskRF(n_estimators=100, max_depth=10, ignore_empty_partitions=True)  # Tune hyperparameters as needed
rf = RandomForestRegressor(**rf_params)
rf.fit(X_train, y_train)

In [ ]:
# Save the model
dump(rf, output_model_filename)

## 7. Get Some Simple Metrics

In [ ]:
rf = load(output_model_filename)

In [ ]:
# Predict on test set
y_pred = rf.predict(X_test)

In [ ]:
metrics_metadata = [
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    mean_absolute_percentage_error,
    median_absolute_error,
    max_error,
    explained_variance_score
]

for skmetric in metrics_metadata:
    print(f'{skmetric.__name__}: {skmetric(y_test, y_pred):.4f}')

In [ ]:
# Scatter plot of predictions vs. true values
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.7, edgecolors="k")
plt.plot([0, 100], [0, 100], color="red", linestyle="--", linewidth=2)  # Reference line

# Add polyfit regression line
poly_coeffs = np.polyfit(y_test, y_pred, deg=1)  # Fit a 1st-degree polynomial (linear fit)
poly_eq = np.poly1d(poly_coeffs)  # Create polynomial function
y_fit = poly_eq(y_test)  # Compute fitted values

plt.plot(y_test, y_fit, color="green", linewidth=2, label=f"Polyfit (y={poly_coeffs[0]:.2f}x + {poly_coeffs[1]:.2f})")

plt.xlabel("True Values")
plt.ylabel("Predicted Values")
plt.title(f"Random Forest Predictions vs. Truth (R²={r2_score(y_test, y_pred):.2f})")
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.grid(True)
plt.show()

## 8. Get Feature Importance

In [ ]:
# Get feature importances
importances = rf.feature_importances_

analysis_columns = col_names.copy()
analysis_columns.remove(target_col)

# Create DataFrame for better visualization
importance_df = pd.DataFrame({"Feature": analysis_columns, "Importance": importances})
importance_df = importance_df.sort_values(by="Importance", ascending=False)

# Print feature importance
print(importance_df)
#print(importance_df.tail(40))
#print(importance_df.tail(20).sort_values(by="Feature"))
#print(importance_df.head(60))

# Plot feature importance
plt.figure(figsize=(6, 4))
plt.barh(importance_df["Feature"], importance_df["Importance"], color="skyblue")
plt.xlabel("Importance Score")
plt.ylabel("Feature")
plt.title("Feature Importance in Random Forest: Mean Decrease in Impurity (MDI)")
plt.gca().invert_yaxis()  # Highest importance at top
plt.show()

## 9. Perform Inference Over Rasters

In [ ]:
def get_band_descriptions(raster_filename):
    band_descriptions = []
    dataset = gdal.Open(raster_filename)
    for i in range(1, dataset.RasterCount + 1):
        band_metadata = dataset.GetRasterBand(i).GetMetadata()
        band_descriptions.append(band_metadata['Name'])
    return band_descriptions

def raster_to_df(raster_filename):
    
    # open raster
    raster = rxr.open_rasterio(raster_filename)
    raster.name = raster_filename
    
    # get band descriptions
    band_descriptions = get_band_descriptions(raster_filename)
    raster_array = raster.values    
    raster_df = pd.DataFrame(raster_array.reshape(raster_array.shape[0], -1).T, columns=band_descriptions)

    # Get the coordinates (x, y)
    x_coords, y_coords = np.meshgrid(raster.coords['x'].values, raster.coords['y'].values)
    raster_df['x'] = x_coords.flatten()
    raster_df['y'] = y_coords.flatten()

    # Add the idx of the positions from 
    raster_df['x_idx'] = np.round(raster_df['x']).astype(int)
    raster_df['y_idx'] = np.round(raster_df['y']).astype(int)

    return raster_df

def open_rasters_to_df(raster_path, output_dir):
    
    # make output dir
    os.makedirs(output_dir, exist_ok=True)
    
    # if the filename does not exist, get the parquet of the bands
    output_filename = os.path.join(output_dir, f'{"_".join(Path(raster_path).parts[-5:-1])}.parquet')
    if not os.path.isfile(output_filename):
        df_list = list()
        raster_file_paths = glob(raster_path)
        for raster_filename in raster_file_paths:
            print(raster_filename)
            df_list.append(raster_to_df(raster_filename))

        # clean df
        raster_df = pd.concat(df_list, axis=1).reset_index(drop=True)
        raster_df = raster_df.loc[:, ~raster_df.columns.duplicated()]

        # save parquet to avoid processing it again
        raster_df.to_parquet(output_filename)
    
    else:
        raster_df = pd.read_parquet(output_filename)
    
    return raster_df

In [ ]:
raster_df = open_rasters_to_df(raster_data_path, output_dir)
raster_df

## 10. Perform Inference with the Model 

In [ ]:
# order the bands as the model is expecting them
# y_predictions = rf.predict(raster_df[col_names + ['x', 'y']]#.drop([target_col], axis=1, errors='ignore'))
# y_predictions.shape
y_predictions = rf.predict(raster_df[analysis_columns + ['x', 'y', 'x_idx', 'y_idx']].drop([target_col, 'x', 'y', 'x_idx', 'y_idx'], axis=1, errors='ignore'))
y_predictions.shape

In [ ]:
predictions_reshaped = np.round(y_predictions.reshape(4800, 4800))

In [ ]:
#import matplotlib.colors as mcolors

# Create a custom brown-to-green colormap
colors = [(0.82, 0.71, 0.55), (0.0, 0.5, 0.0)]  # Brown to Green RGB
n_bins = 100  # Number of bins for the colormap
cmap_name = 'brown_to_green'
brown_to_green = LinearSegmentedColormap.from_list(cmap_name, colors, N=n_bins)

# Plot the data
plt.figure(figsize=(10, 8))

#norm = mcolors.Normalize(vmin=1, vmax=100)
#colored_image = brown_to_green(predictions_reshaped)  # Apply colormap
#colored_image[predictions_reshaped == 0] = [0.5, 0.5, 0.5, 1]  # Set 0s to gray (RGBA format)
 
#imshow_obj = plt.imshow(colored_image)
imshow_obj = plt.imshow(predictions_reshaped, cmap=brown_to_green, vmin=1, vmax=100)
plt.colorbar(imshow_obj, label='Prediction Value', ticks=[1, 50, 100])

# Add title
plt.title("Prediction with Brown to Green Colorbar (1 to 100)")
plt.show()

In [ ]:
def save_output_raster(raster_data_path, prediction, output_dir):
    
    # get one of the rasters for reference
    image = rxr.open_rasterio(glob(raster_data_path)[0])
    image = image.drop(
        dim="band",
        labels=image.coords["band"].values[1:],
    )
    output_filename = os.path.join(output_dir, f'{"_".join(Path(raster_data_path).parts[-5:-1])}.tif')

    # save prediction in raster
    prediction = xr.DataArray(
        np.expand_dims(prediction, axis=0),
        name='',
        coords=image.coords,
        dims=image.dims,
        attrs=image.attrs
    )
    # Add metadata to raster attributes
    prediction.attrs['long_name'] = (output_filename)

    # Set nodata values on mask
    nodata = prediction.rio.nodata

    prediction = prediction.where(image != nodata)
    nodata=250
    prediction.rio.write_nodata(nodata, encoded=True, inplace=True)

    # Save output raster file to disk
    prediction.rio.to_raster(
        output_filename,
        BIGTIFF="IF_SAFER",
        compress='LZW',
        driver='GTiff',
        dtype='uint8'
    )
    return

In [ ]:
# save raster
save_output_raster(raster_data_path, predictions_reshaped, output_dir)

## 11. Get Force Plots

In [ ]:
pixel_id = (0, 0)

In [ ]:
# Explain the model's predictions using SHAP
shap.initjs()  # Load JS visualization support
explainer = shap.TreeExplainer(rf)

In [ ]:
raster_df_summarized = raster_df[analysis_columns + ['x', 'y', 'x_idx', 'y_idx']].drop([target_col, 'x', 'y'], axis=1, errors='ignore')
pixel = raster_df_summarized.loc[
    (raster_df_summarized["x_idx"] == pixel_id[0]) & (raster_df_summarized["y_idx"] == pixel_id[1])].drop([target_col, 'x_idx', 'y_idx'], axis=1, errors='ignore')
pixel

In [ ]:
# Compute SHAP values for this pixel
shap_values = explainer.shap_values(pixel)

In [ ]:
# Plot SHAP force plot for the **single pixel**
shap.force_plot(
    explainer.expected_value,  # Base value
    shap_values[0],  # SHAP values for this pixel
    pixel[0],  # Feature values (spectral bands)
    feature_names=[f"Band {i+1}" for i in range(pixel.shape[1])]  # Naming spectral bands
)

In [ ]:
def plot_force(explainer, idx, to_predict, df):
  single_prediction = explainer(to_predict.iloc[idx, :])
  print(f'Model prediction: {df.iloc[idx]["water"]}')
  shap.force_plot(explainer.expected_value[1],
                    single_prediction.values[:, 1],
                    to_predict.iloc[idx, :],
                    # df_to_predict.to_numpy()[0,:],
                    feature_names=shortened_features,
                    matplotlib=True, 
                    show=True)

In [ ]:
plot_force(explainer, 0, predicted_high_ndvi_water_no_water, predicted_high_ndvi_water)

## 12. Visualize Raster

In [ ]:
# will leave it like this for now

# from localtileserver import get_leaflet_tile_layer, TileClient
# from ipyleaflet import Map

# First, create a tile server from local raster file
# client = TileClient(raster_file_paths[0])

# Create ipyleaflet tile layer from that server
# t = get_leaflet_tile_layer(client)

# m = Map(center=client.center(), zoom=client.default_zoom)
# m.add(t)
# m